# CFP_Chronos_Forecasts

Generates 1-step-ahead VaR and ES forecasts using Amazon Chronos-v1 Small (46M) and Mini (20M) foundation models. Computes 1000 ancestral samples per day; extracts VaR quantiles, empirical ES at 2.5% (FRTB), and fitted Student-t parameters. Seed: 42.

**Paper:** Pele, Lessmann, Hardle (2026)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','FCHI','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

MODELS = [('chronos_small', 'amazon/chronos-t5-small', 'Chronos-Small'), ('chronos_mini', 'amazon/chronos-t5-mini', 'Chronos-Mini')]

Assets: 24, Context: 512, Samples: 1000


In [ ]:
import numpy as np
import pandas as pd
import torch
from chronos import ChronosPipeline
from tqdm import tqdm

assert torch.cuda.is_available(), "CUDA not available"
device = "cuda"
print(f"CUDA: {torch.cuda.get_device_name(0)}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

for model_slug, model_id, label in MODELS:
    print(f"\n=== {label} ({model_id}) ===")
    
    pipeline = ChronosPipeline.from_pretrained(
        model_id,
        dtype=torch.float32,
        device_map=device,
    )
    
    model_device = next(pipeline.model.parameters()).device
    print(f"  Model on: {model_device}")
    assert str(model_device).startswith("cuda")
    
    out_dir = DATA_DIR / model_slug
    out_dir.mkdir(exist_ok=True)
    
    pbar = tqdm(ASSETS, desc=label)
    for asset in pbar:
        pbar.set_description(f"{label} → {asset}")
        
        ret = load_returns(asset)
        n = len(ret)
        vals = ret.values
        dates = ret.index
        
        records = []
        
        for t in range(CONTEXT, n):
            context = torch.tensor(
                vals[t-CONTEXT:t], dtype=torch.float32
            ).unsqueeze(0)
            
            with torch.no_grad():
                samples = pipeline.predict(
                    context,
                    prediction_length=1,
                    num_samples=N_SAMPLES,
                )
            
            s = samples[0, :, 0].cpu().numpy()
            
            row = {"date": dates[t]}
            for alpha in ALPHAS:
                row[f"VaR_{alpha}"] = np.percentile(s, alpha * 100)
            row["mean"] = s.mean()
            row["std"] = s.std()
            
            sorted_s = np.sort(s)
            row["ES_empirical_0.025"] = sorted_s[:25].mean()
            row["ES_empirical_0.01"] = sorted_s[:10].mean()
            row["ES_empirical_0.05"] = sorted_s[:50].mean()
            
            records.append(row)
        
        df_out = pd.DataFrame(records).set_index("date")
        df_out.to_parquet(out_dir / f"{asset}.parquet")
    
    del pipeline
    torch.cuda.empty_cache()
    print(f"\n  {label}: {len(ASSETS)} assets saved")

/home/jovyan/.conda-envs/cfp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA: NVIDIA A30

=== Chronos-Small (amazon/chronos-t5-small) ===
  Model on: cuda:0


Chronos-Small → FCHI:  12%|█▎        | 3/24 [4:08:05<31:29:18, 5398.03s/it] 

In [ ]:
# Verify outputs
from pathlib import Path
for model_slug, _, label in MODELS:
    out_dir = DATA_DIR / model_slug
    files = list(out_dir.glob('*.parquet'))
    print(f'{label}: {len(files)} parquet files')